# 01 — Baseline reproduction (W2V + GAT + MHA)

This notebook reproduces the STGAN paper's baseline configuration on DARPA TRACE under deterministic conditions.

## What is verified
The Word2Vec + GAT + Multi-Head-Attention configuration is evaluated on the test set using the pre-trained weights stored in `trained_weights/trace/baseline_w2v_gat_mha/`. The evaluation script is `eval_trace.py` with paper-faithful neighbour sampling (`num_neighbors=[-1, -1]`, `batch_size=26000`) and the deterministic settings established in `utils/det_utils.py` (`seed=42`, `cudnn.deterministic=True`).

## Expected outcome
The reported F1 score should be approximately **0.9875**, which is the deterministic baseline reproduced in this work. The published paper's claim is F1 = 0.9916, computed with the same `helper()` evaluation function used here.

## Reproducibility note
The training itself is not re-executed in this notebook. The 50 epochs of training weights are already saved under `trained_weights/trace/baseline_w2v_gat_mha/`. To re-train from scratch, refer to `train/train_trace.py` and the README.

## Prerequisites
- `00_setup_and_data.ipynb` completed.
- W2V model present at `train/word2vec_trace.model`.
- 50-epoch trained weights present at `trained_weights/trace/baseline_w2v_gat_mha/`.

## Next
`02_C1_embeddings.ipynb` — comparative study of semantic embeddings (BERT, EmbeddingGemma).

In [ ]:
import os
from pathlib import Path

# Pour le reproducteur : adapter cette ligne au dossier où le dépôt est extrait.
# Sur la station ESTIN durant le développement : "/home/jovyan/stgan/work/STGAN-master"
# Après téléchargement du dépôt sur une autre machine, mettre par exemple :
#   PROJECT_ROOT = Path.home() / "STGAN-PFE-Abdelli"
PROJECT_ROOT = Path(os.environ.get("STGAN_ROOT", Path.cwd().resolve().parent))

# Vérification
assert (PROJECT_ROOT / "code" / "eval").exists(), f"PROJECT_ROOT incorrect : {PROJECT_ROOT}"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

In [ ]:
print("=== Notebook 01 — Baseline reproduction ===")
print("Objective : run STGAN baseline eval (W2V + GAT + MHA) and verify F1 ≈ 0.9875")

In [3]:
import subprocess, time

script = f"""
cd {PROJECT_ROOT}/code/eval
PYTHONUNBUFFERED=1 PYTHONPATH={PROJECT_ROOT}/code \\
nohup python -u eval_trace.py > /tmp/eval_baseline.log 2>&1 &
echo $! > /tmp/eval_baseline.pid
"""
subprocess.run(["bash", "-c", script], check=True)
time.sleep(3)

pid = open("/tmp/eval_baseline.pid").read().strip()
print(f"✓ Eval lancé en arrière-plan (PID {pid})")
print(f"Temps attendu : ~10-15 min")
!tail -10 /tmp/eval_baseline.log

✓ Eval lancé en arrière-plan (PID 19976)
Temps attendu : ~10-15 min
Evaluation:  64%|██████▍   | 30/47 [02:24<01:25,  5.00s/it]                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [8]:
import os, time

pid = open("/tmp/eval_baseline.pid").read().strip()
print("Attente de la fin de l'eval...")
while os.path.exists(f"/proc/{pid}"):
    time.sleep(30)
    print(".", end="", flush=True)
print("\n✓ Eval terminé\n")
!tail -10 /tmp/eval_baseline.log

Attente de la fin de l'eval...

✓ Eval terminé

Epoch: 43                                                  
Epoch: 44                                                  
Epoch: 45                                                  
Epoch: 46                                                  
Epoch: 47                                                  
Epoch: 48                                                  
Epoch: 49                                                  
True Positives: 67382, False Positives: 908, False Negatives: 791,True Negatives: 1141629
Precision: 0.9867, Recall: 0.9884, Fscore: 0.9875
Accuracy: 0.9986
